<a href="https://colab.research.google.com/github/arses-ui/Barnacles/blob/main/Finetune_model/Single_step_Finetune/kfoldtraining_SFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<font size = 6> Generating Feature Vectors for Object Detection Dataset


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pathlib import Path

dataset_path = Path("/content/drive/MyDrive/yolo_dataset")
labels = sorted(dataset_path.rglob("*labels/*.txt"))  #
print(len(labels))

124


In [ ]:
classes =["barnacle"]
cls_idx = [0]

In [ ]:
import pandas as pd

index = [label.stem for label in labels]
labels_df = pd.DataFrame([], columns =cls_idx, index=index)

In [ ]:
from collections import Counter

for label in labels:
    lbl_counter = Counter()

    with open(label) as lf:
        lines = lf.readlines()

    for line in lines:
        # classes for YOLO label uses integer at first position of each line
        lbl_counter[int(line.split(" ", 1)[0])] += 1

    labels_df.loc[label.stem] = lbl_counter

labels_df = labels_df.fillna(0.0)  # replace `nan` values with `0.0`

<font size =4.5> In the above boxes, we have created a pandas dataframe that specifies the image path and the number of instances of different classes in every image. As there are only barnacles in our case, it will only show one column.

In [ ]:
print(labels_df)

                    0
image1_patch_0000  50
image1_patch_0001  59
image1_patch_0002  78
image1_patch_0003  79
image1_patch_0004  47
...                ..
image2_patch_0057   8
image2_patch_0058   7
image2_patch_0061   9
image2_patch_0062  18
image2_patch_0063  26

[124 rows x 1 columns]


<font size =5>In case you haven't, Install scikit-learn

In [ ]:
!pip install  scikit-learn

<font size= 4.5> Now, I will use the scikit library to generate k splits of the dataset. Due to the size of the data, I limit the splits to just 5, but it can very depending on the size of the dataset and personal preferences.

In [ ]:
import random
from sklearn.model_selection import KFold
random.seed(0)
ksplit= 5
kf =KFold(n_splits=ksplit, shuffle=True, random_state=20)
kfolds = list(kf.split(labels_df))

<font size =4.5>Setting the labels for each dataset assigned during the kfold method as "train" and "val" for each fold in a pandas dataframe(folds_df)

In [ ]:
folds = [f"split_{n}" for n in range(1, ksplit + 1)]
folds_df = pd.DataFrame(index=index, columns=folds)

for i, (train, val) in enumerate(kfolds, start=1):
  folds_df.loc[labels_df.iloc[train].index, f"split_{i}"] = "train"
  folds_df.loc[labels_df.iloc[val].index, f"split_{i}"] = "val"

In [ ]:
fold_lbl_distrb = pd.DataFrame(index=folds, columns=cls_idx)

for n, (train_indices, val_indices) in enumerate(kfolds, start=1):
    train_totals = labels_df.iloc[train_indices].sum()
    val_totals = labels_df.iloc[val_indices].sum()

    # To avoid division by zero, we add a small value (1E-7) to the denominator
    ratio = val_totals / (train_totals + 1e-7)
    fold_lbl_distrb.loc[f"split_{n}"] = ratio

<font size = 5 >Creating directories and dataset YAML files for each split.

In [ ]:
import datetime
import yaml

supported_extensions = [".jpg", ".jpeg", ".png"]

# Initialize an empty list to store image file paths
images = []

# Loop through supported extensions and gather image files
for ext in supported_extensions:
    images.extend(sorted((dataset_path / "images").rglob(f"*{ext}")))

# Create the necessary directories and dataset YAML files
save_path = Path(dataset_path / f"{datetime.date.today().isoformat()}_{ksplit}-Fold_Cross-val")
save_path.mkdir(parents=True, exist_ok=True)
ds_yamls = []

for split in folds_df.columns:
    # Create directories
    split_dir = save_path / split
    split_dir.mkdir(parents=True, exist_ok=True)
    (split_dir / "train" / "images").mkdir(parents=True, exist_ok=True)
    (split_dir / "train" / "labels").mkdir(parents=True, exist_ok=True)
    (split_dir / "val" / "images").mkdir(parents=True, exist_ok=True)
    (split_dir / "val" / "labels").mkdir(parents=True, exist_ok=True)

    # Create dataset YAML files
    dataset_yaml = split_dir / f"{split}_dataset.yaml"
    ds_yamls.append(dataset_yaml)

    with open(dataset_yaml, "w") as ds_y:
        yaml.safe_dump(
            {
                "path": split_dir.as_posix(),
                "train": "train",
                "val": "val",
                "names": classes,
            },
            ds_y,
        )

<font size =5> Copy the labels and images into the respective directory('train' or 'val') for each split.

In [ ]:
import shutil

from tqdm import tqdm

for image, label in tqdm(zip(images, labels), total=len(images), desc="Copying files"):
    for split, k_split in folds_df.loc[image.stem].items():
        # Destination directory
        img_to_path = save_path / split / k_split / "images"
        lbl_to_path = save_path / split / k_split / "labels"

        # Copy image and label files to new directory (SamefileError if file already exists)
        shutil.copy(image, img_to_path / image.name)
        shutil.copy(label, lbl_to_path / label.name)

Copying files: 100%|██████████| 124/124 [00:08<00:00, 13.84it/s]


<font size = 5> Install ultralytics to initialize a pretrained model

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 125.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 100.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 98.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

<Font size = 6> Initializing the model

<font size =4> I use a small model for this task for time and efficiency sake

In [ ]:
from ultralytics import YOLO
weights_path = "yolov8s.pt"  # using small model(11.8 million parameters)

<font size =5> Running a loop to perform 5-fold validation. Results and Weights are stored in seperate sub-directories within a larger directory.

In [ ]:
results = {}

#small number of epochs as a pretty small dataset
batch = 16
project = "Kfold_training_barnacles"
epochs = 20

for k, dataset_yaml in enumerate(ds_yamls):
    model = YOLO(weights_path, task="detect")
    results[k] = model.train(
        data=dataset_yaml, epochs=epochs, batch=batch, project=project, name=f"fold_{k + 1}"
    )  # include any additional train arguments

100%|██████████| 21.5M/21.5M [00:00<00:00, 171MB/s]


Ultralytics 8.3.170 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/yolo_dataset/2025-07-28_5-Fold_Cross-val/split_1/split_1_dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=fold_1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overl

train: Scanning /content/drive/MyDrive/yolo_dataset/2025-07-28_5-Fold_Cross-val/split_1/train/labels... 99 images, 0 backgrounds, 0 corrupt: 100%|██████████| 99/99 [00:00<00:00, 337.63it/s]


train: New cache created: /content/drive/MyDrive/yolo_dataset/2025-07-28_5-Fold_Cross-val/split_1/train/labels.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.7±0.1 ms, read: 24.4±11.9 MB/s, size: 110.4 KB)


val: Scanning /content/drive/MyDrive/yolo_dataset/2025-07-28_5-Fold_Cross-val/split_1/val/labels... 25 images, 0 backgrounds, 0 corrupt: 100%|██████████| 25/25 [00:00<00:00, 132.19it/s]


...

      20/20      6.76G      1.222     0.5888       1.14         89        640: 100%|██████████| 7/7 [00:00<00:00,  9.56it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00,  7.06it/s]

                   all         24        974      0.934      0.826      0.897      0.568



20 epochs completed in 0.013 hours.
Optimizer stripped from Kfold_training_barnacles/fold_5/weights/last.pt, 22.5MB
Optimizer stripped from Kfold_training_barnacles/fold_5/weights/best.pt, 22.5MB

Validating Kfold_training_barnacles/fold_5/weights/best.pt...
Ultralytics 8.3.170 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
Model summary (fused): 72 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 1/1 [00:00<00:00,  6.53it/s]


                   all         24        974      0.934      0.826      0.897      0.568
Speed: 0.1ms preprocess, 0.8ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to Kfold_training_barnacles/fold_5


In [ ]:
!zip -r Kfold_training_barnacles_SFT.zip Kfold_training_barnacles

  adding: Kfold_training_barnacles/ (stored 0%)
  adding: Kfold_training_barnacles/fold_4/ (stored 0%)
  adding: Kfold_training_barnacles/fold_4/args.yaml (deflated 52%)
  adding: Kfold_training_barnacles/fold_4/BoxPR_curve.png (deflated 22%)
  adding: Kfold_training_barnacles/fold_4/train_batch1.jpg (deflated 2%)
  adding: Kfold_training_barnacles/fold_4/confusion_matrix_normalized.png (deflated 36%)
  adding: Kfold_training_barnacles/fold_4/train_batch71.jpg (deflated 5%)
  adding: Kfold_training_barnacles/fold_4/train_batch70.jpg (deflated 4%)
  adding: Kfold_training_barnacles/fold_4/labels_correlogram.jpg (deflated 38%)
  adding: Kfold_training_barnacles/fold_4/train_batch2.jpg (deflated 1%)
  adding: Kfold_training_barnacles/fold_4/labels.jpg (deflated 31%)
  adding: Kfold_training_barnacles/fold_4/BoxR_curve.png (deflated 16%)
  adding: Kfold_training_barnacles/fold_4/BoxP_curve.png (deflated 18%)
  adding: Kfold_training_barnacles/fold_4/val_batch0_pred.jpg (deflated 2%)
  addi

In [ ]:
rm -r /content/Kfold_training_barnacles/